**1. CONVOLUTIONAL AUTOENCODERS TRAINING**

In [1]:
import digitalhub as dh
import pandas as pd
import matplotlib.pyplot as plt

NOME_PROGETTO = "floods"
project = dh.get_project(NOME_PROGETTO)
print(f"Progetto: {project.name}")

Progetto: floods


**SETUP PARAMETERS**

In [6]:
# Parametri Job
job_name = "train_sar_2D_convLstm_v1"                                    
dataset = "Standard" 
epochs = 200
train_sar = True
train_opt = False                                           # Test, Standard, Anomalies*
#handler = pretrain_encoders                                         

parametri = {
    "epochs": epochs, 
    "batch_size": 16, 
    "lr": 1e-4, 
    "weight_decay": 1e-4,      
    "patch_size": 256, 
    "n_images1": 4, "n_channels1": 2,                       # sar
    "n_images2": 4, "n_channels2": 10,                      # opt
    "output_dim": 16,                                       # dimensione spazio latente                                                      
    "workers": 0,
    "job_name": job_name,
    "dataset": dataset,
    "train_sar": train_sar,                                  # train sar encoder
    "train_opt": train_opt,                                  # train opt encoder
    "patience": 10,                                    
    "min_delta": 1e-4,
    "ltae": False,
    "resume": False,
    "time_debug": False                                     # time_debug = True solo per debug, = False per training
}

print(f"PARAMETRI: {parametri}")

# volume -> circa 400 GB dataset Standard
volumi = [
    {
        "volume_type": "ephemeral",
        "name": "volume-spazio-dati",
        "mount_path": "/data",      
        "spec": {"size": "500Gi"}   
    }
]

PARAMETRI: {'epochs': 200, 'batch_size': 16, 'lr': 0.0001, 'weight_decay': 0.0001, 'patch_size': 256, 'n_images1': 4, 'n_channels1': 2, 'n_images2': 4, 'n_channels2': 10, 'output_dim': 16, 'workers': 0, 'job_name': 'train_sar_2D_convLstm_v1', 'dataset': 'Standard', 'train_sar': True, 'train_opt': False, 'patience': 10, 'min_delta': 0.0001, 'ltae': False, 'resume': False, 'time_debug': False}


**BUILD ENVIRONMENT**

In [7]:
encoders_train_func = project.new_function(
    name= f'encoders-Floods_{job_name}_{dataset}_{epochs}',
    kind="python",
    python_version="PYTHON3_10",
    code_src="../src/", 
    handler="train_autoencoders_2D", 
    base_image="pytorch/pytorch:2.1.2-cuda11.8-cudnn8-runtime",
    requirements=["pandas==2.3.3", "numpy==1.26.4", "rasterio==1.4.4", "tqdm==4.70.0", "tifffile==2024.8.30", "torch==2.1.2", "matplotlib==3.10.9", "digitalhub==0.15.11", "digitalhub-runtime-python==0.15.2"]
)

# .run("build") -> scarica immagine, installa requirements, copia intera cartella code_src, crea immagine docker

build = encoders_train_func.run("build", wait=True)
print(f"BUILD: {build.status.state}")

2026-09-21 20:07:28,808 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 55dddf6b5ec04a94beb948c0ba75b0af to finish...
2026-09-21 20:07:33,815 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 55dddf6b5ec04a94beb948c0ba75b0af to finish...
2026-09-21 20:07:38,824 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 55dddf6b5ec04a94beb948c0ba75b0af to finish...
2026-09-21 20:07:43,831 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 55dddf6b5ec04a94beb948c0ba75b0af to finish...
2026-09-21 20:07:48,839 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 55dddf6b5ec04a94beb948c0ba75b0af to finish...
2026-09-21 20:07:53,846 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 55dddf6b5ec04a94beb948c0ba75b0af to finish...
2026-09-21 20:07:58,853 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 55dddf6b5ec04a94beb948c0ba75b0af to finish...
2026-09-21 20

BUILD: COMPLETED


**TRAINING**

In [ ]:
# action job = avvia container, esegue script, libera risorse

run_train_encoders = encoders_train_func.run(
    action="job", 
    parameters=parametri, 
    volumes=volumi, 
    profile="1xv100-shared",                                       # 1x = 1 gpu
    # local_execution= True,                                
    wait=True
)

print(f"Run train_encoders avviato: {run_train_encoders.id}")
print(run_train_encoders.status.state)
print(run_train_encoders.status.message)

2026-09-21 20:08:18,953 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7579a7d2e5964659827140150b682c34 to finish...


2026-09-21 20:08:23,958 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7579a7d2e5964659827140150b682c34 to finish...
2026-09-21 20:08:28,967 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7579a7d2e5964659827140150b682c34 to finish...
2026-09-21 20:08:33,974 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7579a7d2e5964659827140150b682c34 to finish...
2026-09-21 20:08:38,983 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7579a7d2e5964659827140150b682c34 to finish...
2026-09-21 20:08:43,992 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7579a7d2e5964659827140150b682c34 to finish...
2026-09-21 20:08:49,095 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7579a7d2e5964659827140150b682c34 to finish...
2026-09-21 20:08:54,103 - dhcore.digitalhub.entities.run._base.entity - INFO - Waiting for run 7579a7d2e5964659827140150b682c34 to finish...
2026-09-21 20

**PLOTS**

In [ ]:
if train_sar:
    # salvataggio log
    path_s1 = project.get_artifact(f"metrics-s1_{job_name}_{dataset}_{epochs}").download(overwrite=True)
    df_s1 = pd.read_csv(path_s1)

    # plot
    plt.figure(figsize=(8, 5))
    plt.plot(df_s1['epoch'], df_s1['train_loss'], color='blue', label='Train Loss')
    plt.title(f'Training CAE SAR - {job_name}_{dataset}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.grid(True)
    plt.legend()
    plt.show()

else:
    print("train sar non eseguito")

BackendError: No object found.

In [ ]:
if train_opt:
    # salvataggio log
    path_s2 = project.get_artifact(f"metrics-s2_{job_name}_{dataset}_{epochs}").download(overwrite=True)
    df_s2 = pd.read_csv(path_s2)

    # plot
    plt.figure(figsize=(8, 5))
    plt.plot(df_s2['epoch'], df_s2['train_loss'], color='red', label='Train Loss')
    plt.title(f'Training CAE OPT - {job_name}_{dataset}_{epochs}')
    plt.xlabel('Epochs')
    plt.ylabel('Loss (MSE)')
    plt.grid(True)
    plt.legend()
    plt.show()

else:
    print("train opt non eseguito")

train opt non eseguito
